In [ ]:
import pandas as pd
from Bio.Seq import Seq

In [ ]:
x4a_to_mutate='TTGAAAGAAGATAAACCTcGcAAAAGTTTGTTTAATGATGCAGGAAACAAGAAGAATTCAATTAAAATGTGGTTT'
mut_start_coord=214781433
mut_end_coord=214781507
aa_to_mutate='LKEDKPRKSLFNDAGNKKNSIKMWF'

x4a_upstream='CCATGTGGGAGCAATAAATTTCtTGTAACAGAtttctttttctttttttcTGTCAGAT'
x4a_downstream='AGCCCTCGAAGTAAGAAAGTCAG'

print(x4a_upstream + x4a_to_mutate + x4a_downstream) #Should print the original oligo sequence

seqs = []

In [ ]:
codons_ranked_by_usage = {
    "A": ["GCC", "GCT", "GCA", "GCG"],
    "C": ["TGC", "TGT"],
    "D": ["GAC", "GAT"],
    "E": ["GAG", "GAA"],
    "F": ["TTC", "TTT"],
    "G": ["GGC", "GGA", "GGG", "GGT"],
    "H": ["CAC", "CAT"],
    "I": ["ATC", "ATT", "ATA"],
    "K": ["AAG", "AAA"],
    "L": ["CTG", "CTC", "CTT", "TTG", "TTA", "CTA"],
    "M": ["ATG"],
    "N": ["AAC", "AAT"],
    "P": ["CCC", "CCT", "CCA", "CCG"],
    "Q": ["CAG", "CAA"],
    "R": ["CGG", "AGA", "AGG", "CGC", "CGA", "CGT"],
    "S": ["AGC", "TCC", "TCT", "AGT", "TCA", "TCG"],
    "T": ["ACC", "ACA", "ACT", "ACG"],
    "V": ["GTG", "GTC", "GTT", "GTA"],
    "W": ["TGG"],
    "Y": ["TAC", "TAT"],
}

all_coords = list(range(mut_start_coord, mut_end_coord+1))

In [ ]:
def make_oligo_from_dict(codon_dict,upstream=x4a_upstream, downstream=x4a_downstream):
    oligo=upstream

    for key in codon_dict.keys():
        seq=codon_dict[key]
        oligo+=seq
    
    oligo = oligo + downstream

    return oligo

# Make Library A (Combo mutants, 6-bp dels, and SNVs)

## Split sequence to mutate

In [ ]:
split_codons=[x4a_to_mutate[i:i+3] for i in range(0, len(x4a_to_mutate), 3)]

split_dict = {}

for i, char in enumerate(aa_to_mutate):
    aa_pos = char + str(123 + i)
    split_dict[aa_pos] = split_codons[i]

print(split_dict)


## Make All Lys -> Arg

In [ ]:
lys_to_arg_dict=split_dict.copy()

for key in lys_to_arg_dict.keys():
    if 'K' in key:
        print(key)

        lys_to_arg_dict[key] = 'CGG'


print(lys_to_arg_dict)

lys_to_arg_oligo=('BARD1_X4A_IDR_LystoArg_1', 'All Lys to Arg', make_oligo_from_dict(lys_to_arg_dict))

seqs.append(lys_to_arg_oligo)
print(lys_to_arg_oligo)

## Make Each K->A

In [ ]:
from itertools import combinations

def get_all_combinations(positions):
    all_combos = []
    for k in range(1, len(positions) + 1):
        combos = list(combinations(positions, k))
        all_combos.extend(combos)
    return all_combos

In [ ]:
var_count = len(seqs)

In [ ]:
positions = [124, 127, 130, 139, 140, 144]
all_combos = get_all_combinations(positions)

for i, var in enumerate(all_combos):
    lys_to_ala_dict = split_dict.copy()
    print(var)
    modified_aas=[]
    for pos in var:
        key = 'K' + str(pos)
        print(key)
        modified_aas.append(key)
        lys_to_ala_dict[key] = 'GCC'

    var_name=','.join(modified_aas)+ ' to Ala'
    print(lys_to_ala_dict)
    oligo_name = 'BARD1_X4A_IDR_LystoAla' + str(1+ i + var_count)
    seqs.append((oligo_name,var_name, make_oligo_from_dict(lys_to_ala_dict)))
        
print(seqs)


## 6-bp dels

In [ ]:
var_count=len(seqs)

In [ ]:
upper_string = x4a_to_mutate.upper()
del_size = 6
all_del_variants = []
for i in range(0, len(upper_string) - del_size + 1, 3):
    del_var = upper_string[:i] + upper_string[i+del_size:]
    if del_var not in all_del_variants:
        all_del_variants.append(del_var)
    else:
        continue


for i, seq in enumerate(all_del_variants):
    full_oligo = x4a_upstream + seq + x4a_downstream
    seq_name = 'BARD1_X4A_IDR_6bp_del' + str(1+ i + var_count)

    del_tuple = (seq_name, full_oligo)
    print(del_tuple)

    seqs.append(del_tuple)

## FXDA

In [ ]:
var_count = len(seqs)

In [ ]:
fxda_pos=[133, 135, 136]

fxda_dict = {133: 'F',
             135: 'D',
             136: 'A'}

fxda_mutants = {'F133': 'GCC',
                'D135': 'GCC',
                'A136': 'GAG'}

fxda_combos = get_all_combinations(fxda_pos)

In [ ]:
for i, var in enumerate(fxda_combos):
    fxda_lib_dict = split_dict.copy()

    modified_aas=[]
    for pos in var:
        print(pos)
        key = fxda_dict[pos] + str(pos)
        print(key)
        modified_aas.append(key)
        fxda_lib_dict[key] = fxda_mutants[key]

    var_name=','.join(modified_aas) + ' modified'
    print(fxda_lib_dict)
    oligo_name = 'BARD1_X4A_IDR_AAE_' + str(1+ i + var_count)
    seqs.append((oligo_name, var_name, make_oligo_from_dict(fxda_lib_dict)))

print(seqs)

## Make SNVs

In [ ]:
var_count = len(seqs)

In [ ]:
def mutagenize(my_string, amp_f, amp_r):
    """Generate all SNVs, skipping lowercase (PAM) positions."""
    all_variants = []
    for i in range(len(my_string)):
        if my_string[i].isupper():
            for j in ("A", "C", "G", "T"):
                if j != my_string[i]:
                    all_variants.append(amp_f + my_string[:i] + j + my_string[i+1:]+ amp_r)
    return all_variants


In [ ]:
BARD1_X4a_AMP_F='CCATGTGGGAGCAATAAATTTC'
BARD1_X4a_AMP_R='CCCTCGAAGTAAGAAAGTCAG'

sge_oligo='tTGTAACAGATTTCTTTTTCTTTTTTTCTGTCAGATTTGAAAGAAGATAAACCTcGcAAAAGTTTGTTTAATGATGCAGGAAACAAGAAGAATTCAATTAAAATGTGGTTTAG'

In [ ]:
all_snvs = mutagenize(sge_oligo, BARD1_X4a_AMP_F, BARD1_X4a_AMP_R)

for i, snv in enumerate(all_snvs):
    var_name = 'BARD1_X4A_IDR_SNV_' + str(i + 1 + var_count)

    seqs.append((var_name, snv))


print(seqs)

## Output Library A

In [ ]:
lib_a_df = pd.DataFrame(seqs, columns=['seq_name','seq'])

#lib_a_df.to_csv('/Users/ivan/Documents/local_work/20260602_BARD1_X4_IDR_libs/lib_csvs/20260603_BARD1_X4A_IDR_LibA.csv', index=False)

# Make Library B (Site Saturation, ClinVar Vars.)

In [ ]:
lib_b_seqs = []

## ClinVar Variants

There are 5 indels in ClinVar in this region. They are:
* NM_000465.4(BARD1):c.433_435delinsTA (p.Met145fs) (1)
* NM_000465.4(BARD1):c.420_421delinsTT (p.Lys140_Asn141delinsAsnTyr) (2)
* NM_000465.4(BARD1):c.403_404delinsTT (p.Asp135Phe) (3)
* NM_000465.4(BARD1):c.377_378delinsGA (p.Asp126Gly) (4)
* NM_000465.4(BARD1):c.365-20_365-1delinsAA (5)


In [ ]:
ref = x4a_upstream + x4a_to_mutate + x4a_downstream
CDS_OFFSET = len(x4a_upstream)  # position in ref where c.367 starts
CDS_C_START = 367

def c_to_ref(c_pos):
    return CDS_OFFSET + (c_pos - CDS_C_START)

def apply_delins(seq, start, end_inclusive, ins):
    return seq[:start] + ins + seq[end_inclusive + 1:]

clinvar_indels = [
    ('BARD1_X4A_IDR_ClinVarIndels_1',  c_to_ref(377), c_to_ref(378), 'GA'),
    ('BARD1_X4A_IDR_ClinVarIndels_2',  c_to_ref(403), c_to_ref(404), 'TT'),
    ('BARD1_X4A_IDR_ClinVarIndels_3',  c_to_ref(420), c_to_ref(421), 'TT'),
    ('BARD1_X4A_IDR_ClinVarIndels_4',  c_to_ref(433), c_to_ref(435), 'TA'),
]

for name, start, end, ins in clinvar_indels:
    lib_b_seqs.append((name, apply_delins(ref, start, end, ins)))


lib_b_seqs.append(('BARD1_X4A_IDR_ClinVarIndels_5', 'CCATGTGGGAGCAATAAATTTCATGTAACAGAtttcAAATTTGAAAGAAGATAAACCTAGGAAAAGTTTGTTTAATGATGCAGGAAACAAGAAGAATTCAATTAAAATGTGGTTTAGCCCTCGAAGTAAGAAAGTCAG'))
print(lib_b_seqs)

## Full Saturation Libraries

In [ ]:
def make_mutations(region_name,
                   region,
                   region_flanks=[Seq(''), Seq('')],
                   nt_start=0,
                   wt_only=False,
                   incl_wt = False,
                   synonymous=True,
                   stops='TAA',
                   all3ntdeletions=True,
                   mutation_list=False,
                   codons_ranked_by_usage=codons_ranked_by_usage,
                   aa_start=0):

    oligo_array = {}

    if (len(region) / 3 != len(region) // 3) | (nt_start / 3 != nt_start // 3):
        print('Region is not translatable!')

    else:

        if incl_wt:
            oligo_name = region_name + '_WT'
            wt_seq = region_flanks[0] + region + region_flanks[1]
            oligo_array[oligo_name] = wt_seq

        if not wt_only:

            if mutation_list == False:

                seen_del_seqs = set()

                for j in range(0, len(region), 3):

                    aa = region[j:(j + 3)].upper().translate()
                    for aa_to in codons_ranked_by_usage.keys():
                        if aa_to != aa:
                            oligo_name = region_name + '_' + str(aa) + str((nt_start + j) // 3 + 1 + aa_start) + str(aa_to)
                            seq_to_append = \
                                region_flanks[0] + \
                                region[0:j] + Seq(codons_ranked_by_usage[aa_to][0]) + \
                                region[(j + 3):] + \
                                region_flanks[1]
                            oligo_array[oligo_name] = seq_to_append

                    if synonymous:
                        if len(codons_ranked_by_usage[str(aa)]) > 1:
                            oligo_name = region_name + '_' + str(aa) + str((nt_start + j) // 3 + 1 + aa_start) + str(aa)
                            possible_codons = codons_ranked_by_usage[str(aa)].copy()
                            possible_codons.remove(str(region[j:(j + 3)]).upper())
                            seq_to_append = \
                                region_flanks[0] + \
                                region[0:j] + Seq(possible_codons[0]) + \
                                region[(j + 3):] + \
                                region_flanks[1]
                            oligo_array[oligo_name] = seq_to_append

                    if stops:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start + j) // 3 + 1 + aa_start) + 'X'
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(stops) + \
                            region[(j + 3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                    if all3ntdeletions:
                        for k in range(0, 3):
                            if j + k + 3 <= len(region):
                                seq_to_append = \
                                    region_flanks[0] + \
                                    region[0:(j + k)] + \
                                    region[(j + k + 3):] + \
                                    region_flanks[1]
                                seq_str = str(seq_to_append)
                                if seq_str not in seen_del_seqs:
                                    seen_del_seqs.add(seq_str)
                                    oligo_name = region_name + '_' + 'del' + str(nt_start + j + k + 1 + 3 * aa_start)
                                    oligo_array[oligo_name] = seq_to_append

            else:

                for i in range(len(mutation_list)):
                    oligo_name = region_name + '_' + 'variant' + str(i + 1)
                    seq_to_append = region
                    for k, v in enumerate(mutation_list[i]):
                        aa_from = v[0]
                        aa_to = v[-1]
                        pos = int(v[1:-1])
                        j = 3 * (pos - aa_start - 1)
                        aa = region[j:(j + 3)].upper().translate()
                        if aa != aa_from:
                            print('Check mutation list!')
                        else:
                            seq_to_append = \
                                seq_to_append[0:j] + \
                                Seq(codons_ranked_by_usage[aa_to][0]) + \
                                seq_to_append[(j + 3):]
                    seq_to_append = region_flanks[0] + seq_to_append + region_flanks[1]
                    oligo_array[oligo_name] = seq_to_append

    return oligo_array

In [ ]:
def extract_orf(tile, frame):
    """
    Extract the ORF from a tile sequence given the reading frame.

    Parameters
    ----------
    tile : str
        DNA tile sequence.
    frame : int or str
        Base position within a codon at which the tile starts (1, 2, or 3).
        1 = tile starts at the 1st base of a codon
        2 = tile starts at the 2nd base of a codon
        3 = tile starts at the 3rd (last) base of a codon

    Returns
    -------
    orf_nt : str
        Nucleotide sequence trimmed to complete codons, ending after the first
        stop codon (if present).
    orf_aa : str
        Translated amino acid sequence, ending after the first stop codon
        (if present).
    """
    frame = int(frame)
    if frame not in (1, 2, 3):
        raise ValueError(f"frame must be 1, 2, or 3; got {frame}")

    offset = (4 - frame) % 3
    orf_nt = tile[offset:]

    remainder = len(orf_nt) % 3
    if remainder:
        orf_nt = orf_nt[:-remainder]

    orf_aa = str(Seq(orf_nt).translate())

    stop_idx = orf_aa.find('*')
    if stop_idx != -1:
        orf_aa = orf_aa[:stop_idx + 1]
        orf_nt = orf_nt[:(stop_idx + 1) * 3]

    return orf_nt, orf_aa

In [ ]:
def make_tile_mutations(tile_name,
                        tile,
                        tile_amp_f,
                        tile_amp_r,
                        frame,
                        nt_start=0,
                        wt_only=False,
                        synonymous=True,
                        stops='TAA',
                        all3ntdeletions=False,
                        mutation_list=False,
                        codons_ranked_by_usage=codons_ranked_by_usage,
                        aa_start=0):
    """
    Generate a full saturation mutagenesis oligo library for a single tile.

    Extracts the in-frame ORF from the tile, then generates all variants
    (missense, synonymous, stop, 3-nt deletions) using make_mutations.
    Each oligo is returned as: tile_amp_f + variant_tile + tile_amp_r.

    Lowercase bases in the tile are preserved in non-mutagenized positions
    to demarcate fixed edits; only the substituted codon is uppercased
    (as it comes from codons_ranked_by_usage).
    """
    orf_nt, orf_aa = extract_orf(tile.upper(), frame)
    offset = (4 - int(frame)) % 3

    orf_nt_original_case = tile[offset:offset + len(orf_nt)]
    tile_suffix = tile[offset + len(orf_nt):]

    if orf_aa.endswith('*'):
        tile_suffix = orf_nt_original_case[-3:] + tile_suffix
        orf_nt_original_case = orf_nt_original_case[:-3]

    tile_prefix = tile[:offset]

    region_flanks = [
        Seq(tile_amp_f) + Seq(tile_prefix),
        Seq(tile_suffix) + Seq(tile_amp_r),
    ]

    return make_mutations(
        region_name=tile_name,
        region=Seq(orf_nt_original_case),
        region_flanks=region_flanks,
        nt_start=nt_start,
        wt_only=wt_only,
        synonymous=synonymous,
        stops=stops,
        all3ntdeletions=all3ntdeletions,
        mutation_list=mutation_list,
        codons_ranked_by_usage=codons_ranked_by_usage,
        aa_start=aa_start,
    )

In [ ]:
full_saturation = make_tile_mutations('BARD1_X4A_IDR_FullSaturation', x4a_to_mutate, x4a_upstream, x4a_downstream, frame=1, aa_start=122)


In [ ]:
lib_b_var_count = len(lib_b_seqs)

In [ ]:
for i, var in enumerate(full_saturation.keys()):

    var_name = var + '_' +  str(i + 1 + lib_b_var_count)
    var_seq = str(full_saturation[var])

    lib_b_seqs.append((var_name, var_seq))
    

In [ ]:
lib_b_df = pd.DataFrame(lib_b_seqs, columns=['seq_name', 'seq'])
#lib_b_df.to_csv('/Users/ivan/Documents/local_work/20260602_BARD1_X4_IDR_libs/lib_csvs/20260603_BARD1_X4A_IDR_LibB.csv', index=False)

In [ ]:
lib_b_df